Audit response lengths and truncation indicators in completed clinical runs.....


In [1]:
from pathlib import Path
import os
PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(PROJECT_ROOT).startswith('\\\\?\\')):
    PROJECT_ROOT = Path('\\\\?\\' + str(PROJECT_ROOT))


In [2]:
import os
import re
import numpy as np
import pandas as pd

def read_csv_normalized(*args, **kwargs):
    data = pd.read_csv(*args, **kwargs)
    if 'Vignette' in data.columns:
        data['Vignette'] = data['Vignette'].map(lambda value: value.replace('\r\n', '\n') if isinstance(value, str) else value)
    return data
ROOT = str(PROJECT_ROOT)
OUT = os.path.normpath(os.path.join(ROOT, 'analyses/evaluation_robustness/generation_truncation'))
os.makedirs(OUT, exist_ok=True)
MIN_CAP_HITS = 3
RE_ROUND = re.compile('^r(\\d)_completion_tokens__(.+)$')

def read(path):
    h = open(path, encoding='utf-8', errors='replace').readline()
    sep = ';' if h.count(';') > h.count(',') else ','
    return read_csv_normalized(path, sep=sep, engine='python', on_bad_lines='skip')

def cap_stats(series):
    s = pd.to_numeric(series, errors='coerce').dropna()
    if s.empty:
        return (None, 0, 0, np.nan)
    mx = s.max()
    hits = int((s == mx).sum())
    return (int(mx) if hits >= MIN_CAP_HITS else None, hits, len(s), int(mx))
candidates = []
for root, _dirs, files in os.walk(ROOT):
    for f in sorted(files):
        if not f.endswith('.csv'):
            continue
        p = os.path.normpath(os.path.join(root, f))
        try:
            h = open(p, encoding='utf-8', errors='replace').readline()
        except Exception:
            continue
        sep = ';' if h.count(';') > h.count(',') else ','
        cols = [c.strip().strip('\ufeff"') for c in h.strip().split(sep)]
        if 'Judge_score' in cols:
            continue
        panel = any((RE_ROUND.match(c) for c in cols))
        single = 'completion_tokens' in cols and 'baseline_protocol' in cols
        if panel or single:
            candidates.append((p, 'panel' if panel else 'single'))
run_rows, call_rows, panel_rows = ([], [], [])
for path, kind in candidates:
    rel = os.path.relpath(path, ROOT).replace('\\', '/')
    try:
        d = read(path)
    except Exception as e:
        run_rows.append({'run': rel, 'kind': kind, 'status': f'READ ERR {e}'})
        continue
    if 'case_text' in d.columns:
        d = d[d['case_text'].notna() & (d['case_text'].astype(str).str.len() > 20)]
    n = len(d)
    if n == 0:
        continue
    if kind == 'single':
        cap, hits, valid, mx = cap_stats(d['completion_tokens'])
        call_rows.append({'run': rel, 'call': 'single', 'round': '-', 'expert': '-', 'n_calls': valid, 'max_obs': mx, 'cap': cap, 'n_at_cap': hits if cap else 0, 'trunc_pct': round(100 * hits / valid, 1) if cap and valid else 0.0})
        run_rows.append({'run': rel, 'kind': 'single', 'n_cases': n, 'k_experts': 1, 'cap': cap, 'trunc_calls_pct': round(100 * hits / valid, 1) if cap and valid else 0.0, 'full_r1_pct': np.nan, 'zero_vote_cases': np.nan})
        continue
    experts, rounds = (set(), set())
    for c in d.columns:
        m = RE_ROUND.match(c)
        if m:
            rounds.add(int(m.group(1)))
            experts.add(m.group(2))
    experts, rounds = (sorted(experts), sorted(rounds))
    caps, r1_trunc_hits, r1_trunc_calls = ([], 0, 0)
    for r in rounds:
        for e in experts:
            col = f'r{r}_completion_tokens__{e}'
            if col not in d.columns:
                continue
            cap, hits, valid, mx = cap_stats(d[col])
            call_rows.append({'run': rel, 'call': f'r{r}:{e}', 'round': f'r{r}', 'expert': e, 'n_calls': valid, 'max_obs': mx, 'cap': cap, 'n_at_cap': hits if cap else 0, 'trunc_pct': round(100 * hits / valid, 1) if cap and valid else 0.0})
            if cap:
                caps.append(cap)
            if r == 1:
                r1_trunc_calls += valid
                r1_trunc_hits += hits if cap else 0
    votes_cols = [f'r1_emergency__{e}' for e in experts if f'r1_emergency__{e}' in d.columns]
    if votes_cols:
        nv = d[votes_cols].notna().sum(axis=1)
        dist = nv.value_counts().reindex(range(len(experts) + 1), fill_value=0)
        full = int(dist.get(len(votes_cols), 0))
        zero = int(dist.get(0, 0))
        panel_rows.append({'run': rel, 'k_experts': len(votes_cols), 'n_cases': n, **{f'votes_{i}': int(dist.get(i, 0)) for i in range(len(experts) + 1)}})
    else:
        full, zero = (np.nan, np.nan)
    run_rows.append({'run': rel, 'kind': 'panel', 'n_cases': n, 'k_experts': len(experts), 'cap': int(max(caps)) if caps else None, 'caps_seen': '/'.join((str(c) for c in sorted(set(caps)))) if caps else '-', 'trunc_calls_pct': round(100 * r1_trunc_hits / r1_trunc_calls, 1) if r1_trunc_calls else 0.0, 'full_r1_pct': round(100 * full / n, 1) if votes_cols else np.nan, 'full_r1': full if votes_cols else np.nan, 'zero_vote_cases': zero if votes_cols else np.nan})
runs = pd.DataFrame(run_rows)
calls = pd.DataFrame(call_rows)
panels = pd.DataFrame(panel_rows)
runs.to_csv(os.path.normpath(os.path.join(OUT, 'runs.csv')), index=False, encoding='utf-8')
calls.to_csv(os.path.normpath(os.path.join(OUT, 'calls.csv')), index=False, encoding='utf-8')
panels.to_csv(os.path.normpath(os.path.join(OUT, 'panels.csv')), index=False, encoding='utf-8')
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 90)
print(f"прогонов найдено: {len(runs)}  (панельных {sum(runs.kind == 'panel')}, одиночных {sum(runs.kind == 'single')})")
print(f'CSV: {OUT}')


прогонов найдено: 284  (панельных 208, одиночных 76)
CSV: .\analyses\evaluation_robustness\generation_truncation
